[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/llm-course/blob/main/slides/week6/bert_variants_demo.ipynb)

# BERT variants — interactive exploration

**PSYC 51.17: Models of language and communication**  
**Week 6**

---

## Learning objectives

By the end of this session, you will:
1. Compare predictions across BERT variants (RoBERTa, ALBERT, DistilBERT)
2. Benchmark speed and memory efficiency of different architectures
3. Evaluate embedding quality and semantic clustering across models
4. Explore ELECTRA's unique replaced token detection objective

## Setup

In [ ]:
# Install required packages (for Colab)
!pip install -q transformers torch matplotlib numpy

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
from transformers import (
    pipeline,
    AutoModel,
    AutoTokenizer,
    ElectraForPreTraining,
    ElectraTokenizer
)

warnings.filterwarnings('ignore')

print("\u2713 All imports successful!")

## Part 1: Comparing fill-mask predictions across variants

While the original BERT was a breakthrough, many variants have since improved upon its design. **RoBERTa** changed the training recipe, **ALBERT** reduced parameters through sharing, and **DistilBERT** used knowledge distillation to create a smaller, faster model. Let's see how their predictions differ.

In [ ]:
# Define the models we want to compare
model_names = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "ALBERT": "albert-base-v2",
    "DistilBERT": "distilbert-base-uncased"
}

# Load pipelines (this may take a minute to download models)
pipelines = {}
for name, model_id in model_names.items():
    print(f"Loading {name}...")
    pipelines[name] = pipeline("fill-mask", model=model_id)

def compare_predictions(sentence_template):
    print(f"\nSentence: {sentence_template}")
    print(f"{'Model':<12} | {'Top Prediction':<15} | {'Score'}")
    print("-" * 45)
    
    for name, pipe in pipelines.items():
        # Handle different mask tokens (BERT uses [MASK], RoBERTa uses <mask>)
        mask_token = pipe.tokenizer.mask_token
        sent = sentence_template.replace("[MASK]", mask_token)
        
        result = pipe(sent)[0]
        print(f"{name:<12} | {result['token_str']:<15} | {result['score']:.4f}")

# Compare on a few different contexts
compare_predictions("The capital of France is [MASK].")
compare_predictions("The scientist won the Nobel Prize for their work on [MASK].")
compare_predictions("I went to the [MASK] to buy some bread.")

### 💡 Discussion

- Did all models agree on the top prediction? If not, which ones diverged?
- RoBERTa was trained on much more data than BERT. Can you find a sentence where RoBERTa seems more "knowledgeable"?
- ALBERT is much smaller than BERT but often performs similarly. Does it seem as capable in these simple tests?
- Notice the different mask tokens. Why might different models choose different special tokens?

## Part 2: Speed and memory benchmarking

In production environments, we often care about **latency** (how fast is the prediction?) and **memory footprint** (how big is the model?). Let's benchmark our variants to see the trade-offs.

In [ ]:
sentences = ["The quick brown fox jumps over the lazy dog."] * 200
benchmarks = {}

for name, model_id in model_names.items():
    print(f"Benchmarking {name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id)
    
    # Count parameters
    num_params = sum(p.numel() for p in model.parameters())
    
    # Measure inference time
    start_time = time.time()
    with torch.no_grad():
        for sent in sentences:
            inputs = tokenizer(sent, return_tensors="pt")
            _ = model(**inputs)
    end_time = time.time()
    
    benchmarks[name] = {
        "params": num_params / 1e6,  # Millions
        "time": end_time - start_time
    }

print("\nBenchmark Results:")
for name, data in benchmarks.items():
    print(f"{name:<12}: {data['params']:>6.1f}M params, {data['time']:>6.2f}s for 200 sentences")

In [ ]:
# Visualize the trade-offs
names = list(benchmarks.keys())
params = [benchmarks[n]["params"] for n in names]
times = [benchmarks[n]["time"] for n in names]

fig, ax1 = plt.subplots(figsize=(10, 6))

color_params = '#00693e' # Dartmouth Green
ax1.set_xlabel('Model Architecture')
ax1.set_ylabel('Parameters (Millions)', color=color_params)
ax1.bar(names, params, color=color_params, alpha=0.6, label='Model Size')
ax1.tick_params(axis='y', labelcolor=color_params)

ax2 = ax1.twinx()
color_time = '#267aba' # Blue
ax2.set_ylabel('Inference Time (seconds)', color=color_time)
ax2.plot(names, times, color=color_time, marker='o', linewidth=3, markersize=10, label='Latency')
ax2.tick_params(axis='y', labelcolor=color_time)

plt.title('BERT Variants: Size vs. Speed Trade-off', fontsize=14)
fig.tight_layout()
plt.show()

### 💡 Discussion

- Which model is the fastest? Is it also the smallest?
- ALBERT has very few parameters compared to BERT. Why is it not significantly faster? (Hint: think about the number of operations vs. the number of unique weights).
- If you were building a real-time chatbot for a mobile phone, which model would you choose?
- How does DistilBERT achieve its speedup while maintaining most of BERT's performance?

## Part 3: Embedding quality comparison

We can extract a single vector to represent an entire sentence, but the best method depends on the model. **BERT** was trained with Next Sentence Prediction (NSP), so its **[CLS] token** learns a useful sentence-level summary. **RoBERTa** dropped NSP, so its `<s>` token is not optimized for this — **mean pooling** over all tokens works much better. Let's compare.

In [ ]:
sentences = [
    "The cat is sleeping on the mat.",
    "A feline is resting on the rug.",
    "The stock market is volatile today.",
    "Financial indices showed high variance.",
    "The recipe calls for two cups of flour.",
    "Bake the bread at 350 degrees."
]

def get_embeddings(model_id, sentences, pooling='cls'):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id)
    
    inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
        if pooling == 'cls':
            # Use [CLS] token (position 0) — works well for BERT (trained with NSP)
            embeddings = outputs.last_hidden_state[:, 0, :]
        else:
            # Mean-pool over non-padding tokens — better for RoBERTa (no NSP training)
            mask = inputs['attention_mask'].unsqueeze(-1).float()
            embeddings = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
    return embeddings.numpy()

def plot_similarity(embeddings, title, ax):
    # Compute cosine similarity
    norm = np.linalg.norm(embeddings, axis=1, keepdims=True)
    sim_matrix = (embeddings @ embeddings.T) / (norm @ norm.T)
    
    im = ax.imshow(sim_matrix, cmap="YlGnBu", vmin=0.5, vmax=1.0)
    ax.set_title(title, color='#6f42c1') # Purple title
    ax.set_xticks(range(len(sentences)))
    ax.set_yticks(range(len(sentences)))
    labels = [s[:15] + "..." for s in sentences]
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    return im

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

print("Computing embeddings for BERT...")
bert_embs = get_embeddings("bert-base-uncased", sentences, pooling='cls')
plot_similarity(bert_embs, "BERT [CLS] Similarity", axes[0])

print("Computing embeddings for RoBERTa...")
roberta_embs = get_embeddings("roberta-base", sentences, pooling='mean')
im = plot_similarity(roberta_embs, "RoBERTa Mean-Pooled Similarity", axes[1])

fig.colorbar(im, ax=axes, shrink=0.8)
plt.suptitle("Semantic Clustering Across Variants", fontsize=16, color='#00693e')
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### 💡 Discussion

- Do both models successfully group the pairs of similar sentences (cat/feline, market/finance, recipe/bake)?
- Which model shows higher "contrast" between similar and dissimilar pairs?
- We used [CLS] pooling for BERT but mean pooling for RoBERTa. Try switching them — what happens to RoBERTa with `pooling='cls'`? Why does the matrix become nearly all 1s?
- Try adding a sentence that is syntactically similar but semantically different (e.g., "The stock is sleeping on the mat"). How does the model handle it?

## Part 4: ELECTRA's replaced token detection

**ELECTRA** uses a different pre-training task called **Replaced Token Detection (RTD)**. Instead of masking tokens, a small "generator" model replaces some tokens with plausible alternatives, and a "discriminator" model tries to identify which tokens were replaced. This is much more efficient than MLM.

In [ ]:
# Load ELECTRA discriminator
electra_name = "google/electra-small-discriminator"
tokenizer = AutoTokenizer.from_pretrained(electra_name)
model = ElectraForPreTraining.from_pretrained(electra_name)

def detect_fake_tokens(sentence):
    inputs = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0] # Probability of being 'fake'
        probs = torch.sigmoid(logits).numpy()
    
    print(f"Sentence: {sentence}")
    print(f"{'Token':<12} | {'Fake Prob':<10} | {'Status'}")
    print("-" * 40)
    
    for token, prob in zip(tokens, probs):
        if token in ['[CLS]', '[SEP]']: continue
        status = "FAKE" if prob > 0.5 else "REAL"
        print(f"{token:<12} | {prob:<10.4f} | {status}")

# Example 1: A natural sentence
detect_fake_tokens("The chef cooked a delicious meal for the guests.")

print("\n" + "="*40 + "\n")

# Example 2: A sentence with a 'fake' token (replaced 'cooked' with 'ate')
detect_fake_tokens("The chef ate a delicious meal for the guests.")

In [ ]:
# Let's visualize the discriminator's confidence
sentence = "The computer programmed the human to write better code."
inputs = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])[1:-1] # Remove CLS/SEP

with torch.no_grad():
    logits = model(**inputs).logits[0][1:-1]
    probs = torch.sigmoid(logits).numpy()

plt.figure(figsize=(10, 5))
colors = ['#9d162e' if p > 0.5 else '#00693e' for p in probs] # Red for fake, Green for real
plt.bar(tokens, probs, color=colors)
plt.axhline(y=0.5, color='gray', linestyle='--')
plt.ylabel("Probability of being 'FAKE'", color='#9d162e')
plt.title("ELECTRA Discriminator: Identifying 'Fake' Tokens", fontsize=14)
plt.ylim(0, 1)
plt.show()

### 💡 Discussion

- In the second example, did ELECTRA correctly identify "ate" as the fake token? Why might it be suspicious of that word in that context?
- How is this task different from BERT's masked language modeling? Why might it be more efficient?
- Try a sentence where you replace a word with a synonym. Does ELECTRA still flag it as fake?
- What happens if you give it a completely nonsensical sentence?

## Summary

| Part | What we explored | Key insight |
|------|-----------------|-------------|
| Part 1 | Fill-mask comparison | Different tokenizers and training objectives lead to different predictions. |
| Part 2 | Benchmarking | DistilBERT and ALBERT offer significant speed/size advantages with minimal quality loss. |
| Part 3 | Embedding quality | Pooling strategy matters: [CLS] works for BERT (NSP-trained), mean pooling for RoBERTa. |\n| Part 4 | ELECTRA | Replaced token detection is a more efficient pre-training task than MLM. |

## Further exploration

1. **Parameter sharing**: Explore ALBERT's cross-layer parameter sharing in more detail—how does it affect the model's internal representations?
2. **Distillation**: Try to "distill" a small model yourself using a larger BERT model as a teacher on a specific task.
3. **RoBERTa's data**: RoBERTa was trained on much more data than BERT. Try to find sentences where RoBERTa's "world knowledge" clearly exceeds BERT's.
4. **ELECTRA Generator**: Use the ELECTRA generator model to create "fake" sentences and see if you can trick the discriminator.